## Pydantic warmup
a) Create a BaseModel for a User. It should have a required id (integer) and a required name (string). Instantiate the model with valid data and then with invalid data (e.g., a string for id) to see the ValidationError.

b) Create a BaseModel for a Person with the fields name, age, email, favourite pet. Add appropriate validation in each fields. Tips: you can use built-in EmailStr type in pydantic for validating email. Try out your Person class by instantiating it with different types of values for the fields to see proper validations.

c) Use normal python class to replicate what you have created in b), i.e. create a Person class with proper input validation.

In [49]:
from pydantic import BaseModel, ValidationError,Field, EmailStr
from typing import Literal
import re

In [3]:
# create the user BaseModel
class User(BaseModel):
    name: str
    user_id: int

user1 = User(name = "kalle33", user_id = 554)
user1

User(name='kalle33', user_id=554)

In [9]:
# invalid data to test the validation error
try:
    User(name = 444, user_id = "hej")
except ValidationError as err:
    print(err)

2 validation errors for User
name
  Input should be a valid string [type=string_type, input_value=444, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
user_id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='hej', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing


In [18]:
class Person(BaseModel):
    name: str
    age: int = Field(gt = 0, lt = 101)
    email: EmailStr

try:
    Person(name = "Ace", age = 126, email = "M")
except ValidationError as err:
    print(err)

2 validation errors for Person
age
  Input should be less than 101 [type=less_than, input_value=126, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='M', input_type=str]


In [63]:
class Person:
    def __init__(self, name: str, age: int, email: str):
        if not isinstance(name, str):
            raise TypeError(f"Name must be of type str not {type(name)}")

        self.name = name
        self.age = age
        self.email = email

    @property
    def age(self):
        return self._age
    @age.setter
    def age(self, value:int):
        if not isinstance(value, int):
            raise TypeError(f"Age must be of type int not {type(value)} that you have provided")
        if value < 0 or value > 101:
            raise ValueError(f"Age must be between 0 and 100, not {value} that you provided")
        self._age = value
    
    @property
    def email(self):
        return self._email
    @email.setter
    def email(self, value: str):
        if not isinstance(value, str):
            raise TypeError(f"Email must be of type str not {type(value)}")
        if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z]{2,}$", value):
            raise ValueError(f"Invalid email address: {value}")
        self._email = value

In [67]:
try:
    Person(name = "Ace", age = 22, email = "karl")
except ValueError as err:
    print(err)

Invalid email address: karl


In [54]:
try:
    Person(name = 455, age = 22, email = "karl@email.com")
except TypeError as err:
    print(err)

Name must be of type str not <class 'int'>


In [65]:
try:
    Person(name = "Ace", age = "Fesk", email = "karl@email.com")
except TypeError as err:
    print(err)

Age must be of type int not <class 'str'> that you have provided
